# Predictive Pipeline Notebook\n\nThis notebook orchestrates the full end‑to‑end workflow for predicting defect formation energies in perovskites.  It runs the primary pipeline, checks whether the fallback augmentation is required, and if so re‑executes the modelling steps with the augmented feature set.  The total wall‑clock time is recorded in `data/logs/runtime.log`.  All random seeds are fixed to ensure reproducibility.

In [ ]:
import random\nimport numpy as np\nimport json\nimport time\nfrom pathlib import Path\nfrom llmxive_toolkit import paths\nfrom src.pipeline.run import run as run_pipeline\nfrom src.pipeline.run_fallback import main as run_fallback\n\n# ---------------------------------------------------------------------\n# Set deterministic random seeds (must match seeds used in the scripts)\n# ---------------------------------------------------------------------\nRANDOM_SEED = 42\nrandom.seed(RANDOM_SEED)\nnp.random.seed(RANDOM_SEED)\n

In [ ]:
# ---------------------------------------------------------------------\n# Execute the primary pipeline and, if necessary, the fallback pathway\n# ---------------------------------------------------------------------\nstart_time = time.time()\n\n# 1. Run the standard pipeline (download → featurize → validate → split →\n#    train → evaluate → feature importance)\nrun_pipeline()\n\n# 2. Load the performance report to decide whether fallback is required\nperf_path = paths.data('processed', 'performance.json')\nwith perf_path.open() as f:\n    performance = json.load(f)\n\nif performance.get('fallback_needed'):\n    # The fallback driver replaces the baseline feature matrix with the\n    # structurally‑augmented version and reruns the modelling steps.\n    run_fallback()\n\nend_time = time.time()\nelapsed = end_time - start_time\n

In [ ]:
# ---------------------------------------------------------------------\n# Record total runtime\n# ---------------------------------------------------------------------\nlog_path = paths.data('logs', 'runtime.log')\nlog_path.parent.mkdir(parents=True, exist_ok=True)\nlog_path.write_text(f"Total runtime seconds: {elapsed:.2f}\n")\nprint(f"Runtime written to {log_path}")\n